In [0]:
import data.utilities.common as Commonlib
import numpy as np
import pandas as pd
import yaml

from data.procedures.generic.helpers.cal_functions import (
    create_calendar_cal_mo_ext_df,
    create_calendar_dim_df,
    create_calendar_dt_alimt_df,
    create_calendar_dt_df,
    create_calendar_iso_wk_alimt_df,
    create_calendar_iso_wk_df,
    create_calendar_iso_wk_ext_df,
    create_calendar_iso_wk_mo_df,
    create_calendar_mo_df,
    create_calendar_qtr_df,
    create_calendar_tri_df,
    create_calendar_wk_df,
    create_calendar_yr_df,
)
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.functions import col

In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed to show valid configurations only
CONFIG_BASE_PATH = "../../configuration/dynamic_function/"

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "2. Configuration File")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}{subject_area}/{config_file}"

config = Commonlib.get_object_config(config_file_path)
source_config = config.get("source")
bronze_config = config.get(ML.bronze)
silver_config = config.get(ML.silver)

In [0]:
# get medallion object
medallion = Medallion(config)

# get the name of the df generator function from the config file
function_name = source_config["function"]

# define 'generator_function' as generator function from cal_functions.py
generator_function = globals()[function_name]

# run function, store results in pandas_df
if (str(generator_function).find("create_calendar_iso_wk_ext_df") == -1) & (
    str(generator_function).find("create_calendar_cal_mo_ext_df") == -1
):
    pandas_df = generator_function("1960-01-01", "2150-01-01")
else:
    pandas_df = generator_function()

# convert pandas df to Spark df, define medallion.df
medallion.df = spark.createDataFrame(pandas_df)

# cast silver data types
medallion.cast_silver_dtypes_expr()

# Write medallion df to silver
medallion.write_to_delta(load_type=silver_config.get("load_type", "overwrite"), layer=ML.silver)

In [0]:
try:
    # publish to external stage - snowflake
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()
except Exception as e:
    medallion.logger.error(
        f"An error occured while publishing to snowflake via generic_procedure(bronze_to_silver). subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
    )
except KeyError:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )